In [13]:
import pyvisa
import time
    
class Lakeshore336:
    
    def __init__(self):
        self.rm = pyvisa.ResourceManager()
        self.instrument = self.rm.open_resource("TCPIP0::169.254.62.185::7777::SOCKET")
        self.instrument.read_termination = '\r\n'
        self.instrument.write_termination = '\r\n'
    

    def query(self, command):
        return self.instrument.query(command)

    def write(self, command):
        self.instrument.write(command)

    def set_temperature(self, channel, temperature):
        """
        Set the temperature setpoint for a specific channel.

        Parameters:
        channel (int): Channel number (1-4).
        temperature (float): Setpoint temperature in Kelvin.
        """
        self.write(f'SETP {channel},{temperature}')
        print(f"Setpoint for channel {channel} set to {temperature} K")

    def get_sample_temp(self):
        """Calls LK336 to read all temperatures. Parses down to channels A and B"""
        response = self.query(f'KRDG? a')
        return float(response.strip())

    def get_vti_temp(self):
        """Calls LK336 to read all temperatures. Parses down to channels A and B"""
        response = self.instrument.query(f'KRDG? b')
        return float(response.strip())

    def set_sample_temp(self, temperature):
        if temperature > 300:
            return print(f"Setpoint too high. Please set the temp to 300 K or less.")
        else:
            setpoint = self.instrument.query(f"SETP? 1")
            setpoint = float(setpoint.strip())
            if setpoint != float(temperature):
                self.instrument.write(f"SETP 1, {temperature}")
                setpoint = self.instrument.query(f"SETP? 1")
                setpoint = float(setpoint.strip())
                print(f"VTI Heater Limit Set to {setpoint} K")
                
                temp = self.get_sample_temp()
                while temp != setpoint:
                    if temp > (temperature - 10):
                        if (temp -  temperature) < 10:
                            toggle = self.instrument.query("RANGE? 1")
                            if toggle == '0':
                                self.instrument.write(f"RANGE 1, 2")
                                toggle = self.instrument.query("RANGE? 1")
                                toggle = int(toggle.strip())
                                print(f"Sample Heater Range set to {toggle}")
                        else:
                            if toggle != '0':
                                self.instrument.write(f"RANGE 1, 0")
                                toggle = self.instrument.query("RANGE? 1")
                                toggle = int(toggle.strip())
                                print(f"Sample Heater Range set to {toggle}")
                            time.sleep(5)
                            print(f"Waiting for Sample to cool from {temp}K to {setpoint}K")
                    if temp < (temperature - 10):
                        toggle = self.instrument.query("RANGE? 1")
                        if toggle != '2':
                            self.instrument.write(f"RANGE 1, 2")
                            toggle = self.instrument.query("RANGE? 1")
                            toggle = int(toggle.strip())
                            print(f"Sample Heater Range set to {toggle}")
                        time.sleep(5)
                        print(f"Waiting for Sample to warm from {temp}K to {setpoint}K")    
            else:
                print(f"VTI Heater already set to {temperature}")
                    
    def set_vti_temp(self, temperature):
        if temperature > 300:
            return print(f"Setpoint too high. Please set the temp to 300 K or less.")
        else:
            setpoint = self.instrument.query(f"SETP? 2")
            setpoint = float(setpoint.strip())
            if setpoint != float(temperature):
                self.instrument.write(f"SETP 2, {temperature}")
                setpoint = self.instrument.query(f"SETP? 2")
                setpoint = float(setpoint.strip())
                print(f"VTI Heater Limit Set to {setpoint} K")
                
                temp = self.get_vti_temp()
                while temp != setpoint:
                    if temp > (temperature - 10):
                        if (temp -  temperature) < 10:
                            toggle = self.instrument.query("RANGE? 2")
                            if toggle == '0':
                                self.instrument.write(f"RANGE 2, 3")
                                toggle = self.instrument.query("RANGE? 2")
                                toggle = int(toggle.strip())
                                print(f"VTI Heater Range set to {toggle}")
                        else:
                            if toggle != '0':
                                self.instrument.write(f"RANGE 2, 0")
                                toggle = self.instrument.query("RANGE? 2")
                                toggle = int(toggle.strip())
                                print(f"VTI Heater Range set to {toggle}")
                            time.sleep(5)
                            print(f"Waiting for VTI to cool from {temp}K to {setpoint}K")
                    if temp < (temperature - 10):
                        toggle = self.instrument.query("RANGE? 2")
                        if toggle != '3':
                            self.instrument.write(f"RANGE 2, 3")
                            toggle = self.instrument.query("RANGE? 2")
                            toggle = int(toggle.strip())
                            print(f"VTI Heater Range set to {toggle}")
                        time.sleep(5)
                        print(f"Waiting for VTI to warm from {temp}K to {setpoint}K")    
            else:
                print(f"VTI Heater already set to {temperature}")

In [14]:
LK336 = Lakeshore336()

In [8]:
import pyvisa
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("Available VISA Resources:", resources)

Available VISA Resources: ('GPIB0::1::INSTR', 'GPIB0::2::INSTR')
